# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
import os
import json
import gradio as gr
import sqlite3
import pandas as pd
import base64
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import random
from datetime import date, timedelta

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"OpenAI API Key exists and begins {api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
openai = OpenAI()

In [ ]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [6]:
# Connect to OpenRouter using the OpenAI-compatible API and your OpenRouter API key

openrouter_client= OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)


In [ ]:
MODEL1 = "gpt-5.4-mini"
MODEL2 = "openrouter/free"  #"deepseek/deepseek-v4.1-flash"

In [8]:
system_prompt = """You are an expert sales data analyst.

Answer questions only using the sales database provided through your tools or data source, 
 if the information is unavailable, clearly say so instead of guessing.
Provide accurate, concise answers and perform calculations, summaries, comparisons, 
and filtering when users ask about sales, products, dates, regions, retailers, prices, or units sold."""

In [ ]:
# Generate synthetic sales data with the nine-column structure and loads them into SQLite.
# Change NUM_RECORDS to control how many sales records are generated.

NUM_RECORDS = 10_000

rng = random.Random(42)

products = [
    "Men's Apparel",
    "Women's Apparel",
    "Men's Street Footwear",
    "Women's Street Footwear",
    "Men's Athletic Footwear",
    "Women's Athletic Footwear",
]

states_by_region = {
    "Midwest": [
        "Illinois", "Indiana", "Iowa", "Kansas", "Michigan",
        "Minnesota", "Missouri", "Nebraska", "North Dakota",
        "Ohio", "South Dakota", "Wisconsin"
    ],
    "Northeast": [
        "Connecticut", "Maine", "Massachusetts", "New Hampshire",
        "New Jersey", "New York", "Pennsylvania", "Rhode Island",
        "Vermont"
    ],
    "South": [
        "Arkansas", "Kentucky", "Louisiana", "Oklahoma",
        "Tennessee", "Texas"
    ],
    "Southeast": [
        "Alabama", "Delaware", "Florida", "Georgia", "Maryland",
        "Mississippi", "North Carolina", "South Carolina",
        "Virginia", "West Virginia"
    ],
    "West": [
        "Alaska", "Arizona", "California", "Colorado", "Hawaii",
        "Idaho", "Montana", "Nevada", "New Mexico", "Oregon",
        "Utah", "Washington", "Wyoming"
    ],
}

retailers = [
    "Amazon", "Foot Locker", "Kohl's",
    "Sports Direct", "Walmart", "West Gear"
]
sales_methods = ["Online", "In-store", "Outlet"]

start_date = date(2020, 1, 1)
end_date = date(2021, 12, 31)
num_days = (end_date - start_date).days + 1

rows = []

for _ in range(NUM_RECORDS):
    region = rng.choice(list(states_by_region))
    state = rng.choice(states_by_region[region])
    price = rng.randint(15, 110)
    units = rng.randint(5, 150)
    invoice_date = start_date + timedelta(days=rng.randrange(num_days))

    rows.append({
        "Invoice Date": invoice_date.isoformat(),
        "Product": rng.choice(products),
        "Region": region,
        "Retailer": rng.choice(retailers),
        "Sales Method": rng.choice(sales_methods),
        "State": state,
        "Price per Unit": price,
        "Total Sales": price * units,
        "Units Sold": units,
    })

df = pd.DataFrame(rows)

# Basic checks: preserve the expected schema and valid totals.
assert len(df) == NUM_RECORDS
assert (df["Total Sales"] == df["Price per Unit"] * df["Units Sold"]).all()

print(f"Generated {len(df):,} synthetic sales records.")
print(f"Dataset shape: {df.shape}")


In [ ]:
# Create the local SQLite database from the generated data
conn = sqlite3.connect("sales.db")

try:
    df.to_sql("sales", conn, if_exists="replace", index=False)
finally:
    conn.close()

print(f"Saved {len(df):,} records to sales.db")    

The database file is created in the current working directory when connected to SQLite.

In [16]:
DB = "sales.db"

###  Let's create our first TOOL-CALL function to calculate total sales for a specific product in a specific state.

In [31]:
def total_sales(product, state):
    print(f"DATABASE TOOL CALLED: Total sales for {product} in {state}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute(
            '''
            SELECT SUM("Total Sales")
            FROM sales
            WHERE Product = ? AND State = ?
            ''',
            (product, state)
        )

        result = cursor.fetchone()

        return (
            f"Total sales from {product} in {state} is {result[0]}"
            if result[0]
            else "No sales data available"
        )

In [ ]:
total_sales("Women's Street Footwear", "Idaho")

#### 🔍 Sales Database Search Tool
The search_sales() function searches the sales database using optional filters such as product, state, retailer, region, sales method, and invoice date, returning all matching records.

In [33]:
def search_sales(product=None, state=None, retailer=None, region=None, sales_method=None, invoice_date=None):
    print("DATABASE TOOL CALLED: Searching sales", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        query = '''
        SELECT *
        FROM sales
        WHERE 1=1
        '''

        params = []

        if product:
            query += ' AND Product = ?'
            params.append(product)

        if state:
            query += ' AND State = ?'
            params.append(state)

        if retailer:
            query += ' AND Retailer = ?'
            params.append(retailer)

        if region:
            query += ' AND Region = ?'
            params.append(region)

        if sales_method:
            query += ' AND "Sales Method" = ?'
            params.append(sales_method)

        if invoice_date:
            query += ' AND "Invoice Date" = ?'
            params.append(invoice_date)

        cursor.execute(query, params)

        results = cursor.fetchall()

        if not results:
            return "No matching sales data found."

        return "\n".join(str(row) for row in results)


In [ ]:
print(search_sales(product="Men's Street Footwear",retailer="Foot Locker"))

In [21]:
# dictionary structure that's required to describe our function: JSON shema
sales_function = {
    "name": "total_sales",
    "description": "Get the total sales amount for a specific product in a specific state from the sales database.",
    "parameters": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "The product name to search for, for example Men's Street Footwear."
            },
            "state": {
                "type": "string",
                "description": "The state where the sales happened, for example New York."
            }
        },
        "required": ["product", "state"],
        "additionalProperties": False
    }
}

search_function = {
    "name": "search_sales",
    "description": "Search sales data using optional filters like product, state, retailer, region, sales method, or invoice date.",
    "parameters": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "Product name to search for, for example Men's Street Footwear."
            },
            "state": {
                "type": "string",
                "description": "State where the sales happened, for example New York."
            },
            "retailer": {
                "type": "string",
                "description": "Retailer name to filter results, for example Foot Locker."
            },
            "region": {
                "type": "string",
                "description": "Sales region to filter, for example Northeast."
            },
            "sales_method": {
                "type": "string",
                "description": "Sales method to filter, for example Online or Outlet."
            },
            "invoice_date": {
                "type": "string",
                "description": "Invoice date to search for."
            }
        },
        "required": [],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": sales_function},{"type": "function", "function": search_function}]

In [22]:
available_functions = {"total_sales": total_sales,
                       "search_sales": search_sales}

In [23]:
# Let users to generate an image alongside the answer.
def product_image(product,state):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",

#prompt=f"""
             #An image representing {product}
#displayed in a luxury clothing shop
#located in {state}.
 #"""

            prompt=f"An image representing a {product}, displayed inside a luxury clothing shop together with bunch of same {product}, named in shop wall as {state}  in a luxury cloth shop pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [34]:
## Stream responses from the selected OpenAI or openrouter model, handle sales database tool calls, and generate a product image based on the retrieved results if user want.
## Let the user decide whether product image generation is enabled.

def chat(history, model="GPT", generate_image=False):
    messages = [{"role": "system", "content": system_prompt}] + history
    
    if model == "GPT":   
        response = openai.chat.completions.create(model=MODEL1, messages=messages, stream=True, tools=tools)
    else:
        # OpenRouter uses the exact same delivery syntax as your working GPT connection
        response = openrouter_client.chat.completions.create(model=MODEL2, messages=messages, stream=True, tools=tools)
    
    buffer = {"role": "assistant", "content": "", "tool_calls": []}
    tool_answer = ""
    image = None
    
    for chunk in response:
        if not chunk.choices or len(chunk.choices) == 0:
            continue
            
        delta = chunk.choices[0].delta
        
        # Stream text directly to your dashboard interface
        if hasattr(delta, 'content') and delta.content:
            buffer["content"] += delta.content or ""
            yield history + [buffer], image

        # Capture and append tool calls as OpenRouter sends them
        if hasattr(delta, 'tool_calls') and delta.tool_calls:
            for tool_call_chunk in delta.tool_calls:
                if not buffer["tool_calls"]:
                    # Create the base object container
                    buffer["tool_calls"].append(tool_call_chunk)
                else:
                    # Accumulate JSON argument string data piece by piece safely
                    if tool_call_chunk.function and tool_call_chunk.function.arguments:
                        buffer["tool_calls"][0].function.arguments += tool_call_chunk.function.arguments
        
        # Execute the database tools only when the API specifies it is complete
        if chunk.choices[0].finish_reason == "tool_calls":
            tool_call = buffer["tool_calls"][0]
            
            # Runs your exact database lookup, returns strings and extracts state metrics
            responses, product, state = handle_sales_tool_call(tool_call)
            
            messages.append(buffer)
            messages.extend(responses)
            
            # Generate the visualization only when verified items match
            image = product_image(product, state) if generate_image and product else None
            
            if model == "GPT":   
                next_response = openai.chat.completions.create(model=MODEL1, messages=messages, stream=True)
            else:
                next_response = openrouter_client.chat.completions.create(model=MODEL2, messages=messages, stream=True)
                
            for next_chunk in next_response:
                if next_chunk.choices and len(next_chunk.choices) > 0 and next_chunk.choices[0].delta.content:
                    tool_answer += next_chunk.choices[0].delta.content or ""
                    yield history + [{"role": "assistant", "content": tool_answer}], image

             

In [35]:
# Execute the AI-selected sales tool with its arguments, return the database results to the model, and extract product and state details for image generation.

def handle_sales_tool_call(tool_call):

    responses = []
    product = None

    function_name = tool_call.function.name

    function_to_call = available_functions.get(function_name)

    arguments = json.loads(tool_call.function.arguments)

    product = arguments.get("product")
    state = arguments.get("state")

    sales_data = function_to_call(**arguments)

    responses.append({
        "role": "tool",
        "content": sales_data,
        "tool_call_id": tool_call.id
    })

    return responses, product,state

In [26]:
# Transcribe user-provided audio into text using OpenAI's speech-to-text model.

def transcribe(audio_file):
    if audio_file is None:
        return ""
    with open(audio_file, "rb") as f:
        transcription = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe", 
            file=f
        )
    return transcription.text

In [ ]:
# Build the Gradio interface with aligned chatbot, image, and control panels.

# remove js=force_dark_mode inside with gr.Blocks(js=force_dark_mode) as ui, if you dont wanna use dark mode.
force_dark_mode = """
function() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.replace(url.href);
    }
}
"""

with gr.Blocks(js=force_dark_mode) as ui:

    # Keep the Chatbot and Image panels side by side.
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(height=500, type="messages")
            entry = gr.Textbox(
                        label="Chat with our code expert:",
                        placeholder="Ask a question...",
                        submit_btn="Send",
                        scale=8)

            # Place model selection and image generation click box below the Chatbot.
            with gr.Row():
                ai_model = gr.Dropdown(
                    ["GPT", "OpenRouter"],
                    label="Select Model",
                    value="GPT",
                    scale=1
                )

                generate_image = gr.Checkbox(
                    label="Generate an image with my answer",
                    value=False,
                    scale=1
                )

        with gr.Column(scale=1):
            
            image_output = gr.Image(height=500)

            with gr.Row():
             microphone = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label=None,
            show_label=False,
            scale=1)


            # Place Clear directly below the Image panel.
            clear = gr.Button("Clear")


    # Show three examples: one general question and two sales-tool questions.
    gr.Examples(
        examples=[
            ["Which state had the highest total sales for Women's Apparel sold online?"],
            ["What were the total sales of Men's Street Footwear in New York?"],
            ["Find sales records for Men's Apparel in California sold by Foot Locker through the Online method."]
        ],
        inputs=entry,
        label="Try these example questions"
    )

    # Add the submitted question to the conversation history.
    def do_entry(message, history):
        history = history or []
        history += [{"role": "user", "content": message}]
        return "", history, None

    # Submit questions by pressing Enter or clicking Send.
    entry.submit(
        do_entry,
        inputs=[entry, chatbot],
        outputs=[entry, chatbot, microphone]
    ).then(
        chat,
        inputs=[chatbot, ai_model, generate_image],
        outputs=[chatbot, image_output]
    )

    # Convert microphone audio into text.
    microphone.change(
        transcribe,
        inputs=[microphone],
        outputs=[entry]
    )

    # Clear the conversation, image, textbox, and microphone.
    clear.click(
        lambda: ([], None, "", None),
        inputs=None,
        outputs=[chatbot, image_output, entry, microphone],
        queue=False
    )

# Launch Gradio and open the interface in the browser.
ui.launch(inbrowser=True)